# Part 10 · Notebook 08 — Serving, drift monitoring and shadow mode

**Sessions:** S15–S16 (Deployment, drift & retraining · ML project defense) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Measure distribution drift with the Population Stability Index, and see how noisy it is on small windows.
2. Monitor a feature over time and find when a known shift is detected.
3. Compute a live feature one tick at a time, identical to its batch version.
4. Compare a champion and a challenger in shadow mode.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()

## 1. The Population Stability Index

Cut the training distribution into 10 quantile bins (outer edges ±∞), compute the share of training values `e` and of live values `a` in each bin (+1e-6 against empty bins), and sum `(a − e)·ln(a/e)`. Rules of thumb: below 0.1 stable, 0.1–0.25 watch, above 0.25 drift.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def psi(expected, actual, bins=10):
    expected, actual = np.asarray(expected, float), np.asarray(actual, float)
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected) + 1e-6
    a = np.histogram(actual, edges)[0] / len(actual) + 1e-6
    return ...                                                       # ✍️

rng = np.random.default_rng(0)
ref, same, shifted = rng.normal(size=2000), rng.normal(size=2000), rng.normal(0.5, 1, 2000)
cases = [(ref, same), (ref, shifted), (ref, rng.normal(size=60))]
mine = [p.attempt(psi, *c) for c in cases]
mine = p.check("psi", mine, [p.psi(*c) for c in cases])
small = [p.psi(ref, rng.normal(size=60)) for _ in range(200)]
print(f"same distribution: PSI {mine[0]:.3f};  shifted by 0.5σ: {mine[1]:.3f};  "
      f"60 values, no drift at all: mean {np.mean(small):.3f}, above 0.25 in {np.mean(np.array(small) > 0.25):.0%} of 200 tries")

A half-sigma shift crosses the drift line; so does pure sampling noise on a small window. PSI on 60 values averages well above its large-sample value and "drifts" in a sizeable share of tries. Use windows large enough for the number of bins, or a test with a p-value (KS).

## 2. A monitor on a known shift

Two features, trained on 1,000 stationary days. The live period has 600 days; from live day 300 on, feature `a` shifts by 0.75σ. `p.drift_table` computes the PSI of each feature per block of live days (and a KS p-value for `a`); `p.first_alarm` returns the first block with any PSI above 0.25.

In [ ]:
rng = np.random.default_rng(3)
train = pd.DataFrame({"a": rng.normal(size=1000), "b": rng.normal(size=1000)})
live = pd.DataFrame({"a": np.r_[rng.normal(size=300), rng.normal(0.75, 1, 300)], "b": rng.normal(size=600)},
                    index=pd.RangeIndex(1, 601, name="live day"))
for window in (30, 100):
    table = p.drift_table(train, live, window)
    alarms = (table[["a", "b"]] > 0.25).any(axis=1)
    print(f"blocks of {window} days: first alarm on live day {p.first_alarm(table)};  "
          f"false alarms before day 300: {int(alarms[table.index <= 300].sum())} of {int((table.index <= 300).sum())} blocks")
p.drift_table(train, live, 100).round(3)

Small blocks raise alarms before anything has changed; larger blocks are quiet until the shift, then fire. Pick the window from the false-alarm rate you can live with, and confirm with a second statistic before retraining.

## 3. Train/serve parity

In research, features are computed on whole columns (`rolling(20)`); live, one tick arrives at a time. The live version must give **the same numbers**. Keep running sums `s` and `s2` of the last `window` values: when the buffer is too long, remove the oldest value from both sums.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from collections import deque

class MyStreamingRolling:
    def __init__(self, window):
        self.window, self.buf = window, deque()
        self.s = self.s2 = 0.0

    def update(self, x):
        self.buf.append(x)
        self.s += x
        self.s2 += x * x
        if len(self.buf) > self.window:
            old = ...                                                # ✍️ drop the oldest value
            self.s -= ...                                            # ✍️
            self.s2 -= ...                                           # ✍️
        if len(self.buf) < self.window:
            return np.nan, np.nan
        n = self.window
        mean = self.s / n
        return mean, float(np.sqrt(max((self.s2 - n * mean * mean) / (n - 1), 0.0)))

values = np.log(p.regime_market()["close"]).diff().dropna().to_numpy()[:400]
live = MyStreamingRolling(20)
mine = p.check("StreamingRolling", p.attempt(lambda: np.array([live.update(v) for v in values])),
               pd.Series(values).rolling(20).agg(["mean", "std"]).to_numpy())
print(f"live (O(1) per tick) vs pandas (batch): max difference {np.nanmax(np.abs(mine - pd.Series(values).rolling(20).agg(['mean', 'std']).to_numpy())):.1e}")

Equal to machine precision. (Running sums can lose precision over millions of ticks; recompute them from the buffer now and then.) The parity test belongs in CI: the same input through both paths, the same output.

## 4. Shadow mode

A challenger runs next to the live champion on the same data; its positions are logged, only the champion trades. Promote only with enough days **and** a clear margin (`p.shadow_compare`: at least 60 days and a Sharpe margin of 0.3).

In [ ]:
mk = p.regime_market()
r = np.log(mk["close"]).diff()
champion, challenger = p.momentum_side(mk["close"], 20), p.momentum_side(mk["close"], 60)
rows = {f"{n} days": p.shadow_compare(r.iloc[:n], champion.iloc[:n], challenger.iloc[:n]) for n in (40, 400, 1500)}
pd.DataFrame(rows).T

After 40 days nothing can be decided (the 60-day signal has barely started). After 400 days the challenger is ahead, but by less than the margin: no promotion. After 1,500 days it clears the margin, though both are weak strategies: shadow mode picks the better of two, it does not make either good.

## S16 · The project defense

The ML project defense (milestone M6) is a report, not a notebook. It must show, in this order: the baseline; the data (bars, events, point-in-time features, truncation test); the labels and weights; purged CV results with the audit; every trial counted for the DSR; walk-forward results after costs against the primary rule; the monitoring plan (drift, parity, shadow period). Clinic W3 in the labs builds it.

## Wrap-up

* PSI needs enough data per bin; small windows cry wolf.
* Live features must equal research features: test the parity.
* Shadow mode first, promotion by rule.
* Graded version: `labs/part10/week36_strategies` (`psi`, `drift_table`, `StreamingRolling`, `shadow_compare`).